# Experiment 12: V3 Prompt vs Human (3 Students)

This notebook runs the V3 prompt on the same three students used in the human annotation study: 10155, 14476, and 14475.

It does two things:
1. Generate V3 KC-gap annotations with Gemini 2.5 Flash using the baseline-only prompt.
2. Compare V3 against the two human raters with the same binary KC agreement metric used in Experiment 11, including per-KC kappa.

Outputs are written to `results/human_validation/` as per-student JSON files, a combined CSV, and comparison summaries.

In [14]:
import json
import os
import time
from pathlib import Path
from statistics import mean
from typing import Dict, List, Set, Tuple

import pandas as pd
from google.genai import types

ROOT = Path.cwd()
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent
    os.chdir(ROOT)

from lib.experiment_utils import create_client, load_best_attempts_df
from lib.llm_batch_analyzer import clean_json_response, format_submissions
from lib.prompts import build_v3_prompt
from utils.constants import GEMINI_API_KEY, PROBLEM_PROMPT_PATH

MODEL_ID = 'gemini-2.5-flash'
SLEEP_SECONDS = 7.0
HUMAN_STUDENTS = [10155, 14476, 14475]
OUTPUT_DIR = Path('results/human_validation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = create_client()
problem_prompts_df = pd.read_csv(PROBLEM_PROMPT_PATH)

print(f'Working directory: {os.getcwd()}')
print(f'Using model: {MODEL_ID}')
print(f'GEMINI_API_KEY loaded: {bool(GEMINI_API_KEY)}')

Working directory: /mnt/d/Projects/kintsugi
Using model: gemini-2.5-flash
GEMINI_API_KEY loaded: True


In [15]:
from sklearn.metrics import cohen_kappa_score, f1_score, precision_score, recall_score

KC_COLUMNS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]

def get_required_kcs(problem_id, problem_prompts_df):
    """Get the list of required KC names for a given problem."""
    row = problem_prompts_df[problem_prompts_df['ProblemID'] == problem_id]
    if row.empty:
        return []
    row = row.iloc[0]
    return [kc for kc in KC_COLUMNS if pd.notna(row.get(kc)) and row.get(kc) == 1]

def get_problem_info(problem_id, problem_prompts_df):
    """Get requirement text and assignment ID for a problem."""
    row = problem_prompts_df[problem_prompts_df['ProblemID'] == problem_id]
    if row.empty:
        return None, None
    row = row.iloc[0]
    return row['Requirement'], row['AssignmentID']

def load_annotations(filepath: str) -> Tuple[str, Dict[str, Set[str]]]:
    path = Path(filepath)
    if not path.exists():
        return path.stem, {}
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    rater_name = data.get('rater', path.stem)
    student_id = data.get('student_id', data.get('studentId', 'unknown'))
    parsed: Dict[str, Set[str]] = {}
    for pid, val in data.get('annotations', {}).items():
        comp_pid = f'{student_id}_{pid}'
        if isinstance(val, dict) and 'gaps' in val:
            gaps = val.get('gaps')
            parsed[comp_pid] = set(gaps) if isinstance(gaps, list) else set()
        elif isinstance(val, list):
            parsed[comp_pid] = set(val)
        else:
            parsed[comp_pid] = set()
        parsed[comp_pid] = {tag for tag in parsed[comp_pid] if tag in KC_COLUMNS}
    return rater_name, parsed

def merge_rater_files(filepaths: List[str], label: str) -> Tuple[str, Dict[str, Set[str]]]:
    merged = {}
    for fp in filepaths:
        _, anns = load_annotations(fp)
        for comp_pid, kcs in anns.items():
            if comp_pid in merged:
                merged[comp_pid].update(kcs)
            else:
                merged[comp_pid] = set(kcs)
    return label, merged

def compute_binary_metrics(y_true, y_pred):
    kappa = cohen_kappa_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    return {'kappa': kappa, 'f1': f1, 'precision': precision, 'recall': recall}

def compute_metrics(name_a: str, anns_a: Dict[str, Set[str]], name_b: str, anns_b: Dict[str, Set[str]], common_pids: List[str]):
    y_a, y_b = [], []
    for pid in common_pids:
        gaps_a, gaps_b = anns_a.get(pid, set()), anns_b.get(pid, set())
        for kc in KC_COLUMNS:
            y_a.append(1 if kc in gaps_a else 0)
            y_b.append(1 if kc in gaps_b else 0)

    metrics = compute_binary_metrics(y_a, y_b)
    metrics['name'] = f'{name_a} vs {name_b}'
    return metrics

def compute_per_kc_metrics(name_a: str, anns_a: Dict[str, Set[str]], name_b: str, anns_b: Dict[str, Set[str]], common_pids: List[str]) -> pd.DataFrame:
    rows = []
    for kc in KC_COLUMNS:
        y_a = []
        y_b = []
        for pid in common_pids:
            gaps_a, gaps_b = anns_a.get(pid, set()), anns_b.get(pid, set())
            y_a.append(1 if kc in gaps_a else 0)
            y_b.append(1 if kc in gaps_b else 0)
        metrics = compute_binary_metrics(y_a, y_b)
        rows.append({
            'KC': kc,
            'Kappa': metrics['kappa'],
            'F1': metrics['f1'],
        })
    return pd.DataFrame(rows)

In [16]:
HUMAN_A_FILES = [
    'dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_10155_1774736175604.json',
    'dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14475_1775593592812.json',
    'dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14476_1775593175294.json',
]
HUMAN_B_FILES = [
    'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_10155_1774820622134.json',
    'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14475_1775593132134.json',
    'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14476_1775691915454.json',
]
V3_OUTPUT_FILES = [
    f'results/human_validation/llm_v3_annotations_{sid}.json' for sid in HUMAN_STUDENTS
]

_, anns_ha = merge_rater_files(HUMAN_A_FILES, 'Human A')
_, anns_hb = merge_rater_files(HUMAN_B_FILES, 'Human B')

target_students = {}
for fpath in HUMAN_A_FILES:
    with open(fpath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    sid = str(data.get('student_id', data.get('studentId')))
    target_students[sid] = sorted(list(data.get('annotations', {}).keys()))

best_attempts_df = load_best_attempts_df()
student_data_by_sid: Dict[str, pd.DataFrame] = {}

for sid_str, pids in target_students.items():
    sid_int = int(sid_str)
    pid_ints = [int(pid) for pid in pids]
    student_df = best_attempts_df[
        (best_attempts_df['SubjectID'] == sid_int) & (best_attempts_df['ProblemID'].isin(pid_ints))
    ].copy()
    if 'Attempt' in student_df.columns:
        student_df = student_df.sort_values(['ProblemID', 'Score', 'Attempt'])
    else:
        student_df = student_df.sort_values(['ProblemID', 'Score'])
    student_df = student_df.drop_duplicates(subset=['ProblemID'], keep='last')
    missing = sorted(set(pid_ints) - set(student_df['ProblemID'].astype(int).tolist()))
    if missing:
        print(f'Warning: student {sid_str} is missing {len(missing)} annotated problems from the dataset: {missing[:10]}')
    print(f'Student {sid_str}: {len(student_df)} submissions loaded for V3')
    student_data_by_sid[sid_str] = student_df

print(f'Total V3 student/problem rows: {sum(len(df) for df in student_data_by_sid.values())}')

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)
Student 10155: 46 submissions loaded for V3
Student 14475: 50 submissions loaded for V3
Student 14476: 50 submissions loaded for V3
Total V3 student/problem rows: 146


In [ ]:
def should_skip_submission(score: float, code: str) -> Tuple[bool, str, List[str]]:
    if score >= 1.0:
        return True, 'Perfect score — no gaps', []

    code_lines = [
        line.strip() for line in code.split('\n')
        if line.strip() and not line.strip().startswith('//')
    ]
    if len(code_lines) <= 3 and not any(keyword in code.lower() for keyword in ['if', 'for', 'while', 'else']):
        return True, 'Placeholder code — no diagnosis', []

    return False, '', []

results = []

for sid_str, student_df in student_data_by_sid.items():
    student_records = {}
    student_raw = {}

    print(f'\n=== Running V3 for student {sid_str} ({len(student_df)} problems) ===')

    for idx, (_, row) in enumerate(student_df.iterrows(), start=1):
        subject_id = int(row['SubjectID'])
        problem_id = int(row['ProblemID'])
        score = float(row['Score'])
        code = row.get('Code', '')
        if pd.isna(code):
            code = ''
        else:
            code = str(code)

        requirement, assignment_id = get_problem_info(problem_id, problem_prompts_df)
        assignment_id = int(assignment_id) if assignment_id is not None and pd.notna(assignment_id) else None
        required_kcs = get_required_kcs(problem_id, problem_prompts_df)

        skip, reason, gaps = should_skip_submission(score, code)
        start_time = time.time()
        raw_text = ''
        parsed = {'reasoning': reason, 'knowledge_gaps': gaps}
        parse_status = 'skipped' if skip else 'ok'

        if not skip:
            prompt = build_v3_prompt(
                problem_id=problem_id,
                requirement=requirement,
                assignment_id=assignment_id,
                required_kcs=required_kcs,
                student_code=code,
                score=score,
            )
            try:
                response = client.models.generate_content(
                    model=MODEL_ID,
                    contents=format_submissions([row.to_dict()]),
                    config=types.GenerateContentConfig(
                        system_instruction=prompt,
                        temperature=0.3,
                        response_mime_type='application/json',
                    ),
                )
                raw_text = response.text if response and response.text else '{}'
                parsed = json.loads(clean_json_response(raw_text))
                reason = parsed.get('reasoning', '')
                gaps = parsed.get('knowledge_gaps', [])
                parse_status = 'ok'
            except Exception as exc:
                raw_text = ''
                parsed = {'reasoning': f'ERROR: {exc}', 'knowledge_gaps': []}
                reason = parsed['reasoning']
                gaps = []
                parse_status = 'error'

        elapsed = round(time.time() - start_time, 3)
        record = {
            'SubjectID': subject_id,
            'ProblemID': problem_id,
            'Score': score,
            'Requirement': requirement,
            'AssignmentID': assignment_id,
            'RequiredKCs': json.dumps(required_kcs),
            'V3_reasoning': reason,
            'V3_knowledge_gaps': json.dumps(gaps),
            'V3_TimeSec': elapsed,
            'ParseStatus': parse_status,
            'RawResponse': raw_text,
            'ParsedResponse': json.dumps(parsed),
        }
        results.append(record)
        student_records[str(problem_id)] = {'gaps': gaps}
        student_raw[str(problem_id)] = {
            'raw_response': raw_text,
            'parsed_response': parsed,
            'parse_status': parse_status,
            'time_sec': elapsed,
            'score': score,
            'assignment_id': assignment_id,
            'required_kcs': required_kcs,
        }

        print(f'  {idx:>3}/{len(student_df)} Problem {problem_id} score={score:.2f} gaps={gaps}')

        if not skip:
            time.sleep(SLEEP_SECONDS)

    student_payload = {
        'rater': 'LLM_Gemini_V3',
        'student_id': sid_str,
        'model_id': MODEL_ID,
        'annotations': student_records,
        'raw_responses': student_raw,
    }
    output_path = OUTPUT_DIR / f'llm_v3_annotations_{sid_str}.json'
    with output_path.open('w', encoding='utf-8') as f:
        json.dump(student_payload, f, indent=2)
    print(f'Saved {output_path}')

results_df = pd.DataFrame(results)
results_csv = OUTPUT_DIR / 'v3_human_annotated_results.csv'
results_df.to_csv(results_csv, index=False)
print(f'\nSaved combined CSV to {results_csv}')
print(f'Rows processed: {len(results_df)}')
print(f'Non-empty gaps: {sum(1 for row in results if row["V3_knowledge_gaps"] != "[]")}')


=== Running V3 for student 10155 (46 problems) ===
    1/46 Problem 1 score=1.00 gaps=[]
    2/46 Problem 3 score=0.81 gaps=['LogicAndNotOr']
    3/46 Problem 5 score=1.00 gaps=[]
    4/46 Problem 12 score=1.00 gaps=[]
    5/46 Problem 13 score=1.00 gaps=[]
    6/46 Problem 17 score=1.00 gaps=[]
    7/46 Problem 20 score=1.00 gaps=[]
    8/46 Problem 21 score=1.00 gaps=[]
    9/46 Problem 22 score=0.36 gaps=['DefFunction', 'LogicCompareNum', 'LogicAndNotOr', 'If/Else']
   10/46 Problem 24 score=0.59 gaps=['LogicCompareNum', 'LogicAndNotOr', 'If/Else']
   11/46 Problem 25 score=1.00 gaps=[]
   12/46 Problem 28 score=0.20 gaps=['LogicCompareNum', 'StringIndex', 'StringConcat', 'StringFormat']
   13/46 Problem 31 score=0.27 gaps=[]
   14/46 Problem 32 score=0.00 gaps=[]
   15/46 Problem 33 score=0.13 gaps=[]
   16/46 Problem 34 score=0.50 gaps=[]
   17/46 Problem 36 score=0.41 gaps=[]
   18/46 Problem 37 score=0.67 gaps=['StringEqual', 'LogicAndNotOr']
   19/46 Problem 38 score=0.73 gaps

In [ ]:
V3_FILES = [str(OUTPUT_DIR / f'llm_v3_annotations_{sid}.json') for sid in HUMAN_STUDENTS]

_, anns_v3 = merge_rater_files(V3_FILES, 'LLM V3')

common_pids = sorted(set(anns_ha.keys()) & set(anns_hb.keys()) & set(anns_v3.keys()))
common_pids = [
    pid for pid in common_pids
    if any(len(rater.get(pid, set())) > 0 for rater in [anns_ha, anns_hb, anns_v3])
]

print(f'Common non-empty problems: {len(common_pids)}')

res_h_h = compute_metrics('Human A', anns_ha, 'Human B', anns_hb, common_pids)
res_a_v3 = compute_metrics('Human A', anns_ha, 'LLM V3', anns_v3, common_pids)
res_b_v3 = compute_metrics('Human B', anns_hb, 'LLM V3', anns_v3, common_pids)
avg_v3_kappa = mean([res_a_v3['kappa'], res_b_v3['kappa']]) if common_pids else 0.0
avg_v3_f1 = mean([res_a_v3['f1'], res_b_v3['f1']]) if common_pids else 0.0

def load_summary_value(summary_path: str, path_keys: List[str]) -> float | None:
    path = Path(summary_path)
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    current = data
    for key in path_keys:
        if not isinstance(current, dict) or key not in current:
            return None
        current = current[key]
    return current if isinstance(current, (int, float)) else None

v1_kappa = load_summary_value('results/human_validation/exp10_baseline_vs_human_metrics.json', ['average_vs_baseline', 'kappa'])
v2_kappa = load_summary_value('results/human_validation/exp11_baseline_v2_vs_human_metrics.json', ['average_vs_baseline', 'kappa'])
v1_f1 = load_summary_value('results/human_validation/exp10_baseline_vs_human_metrics.json', ['average_vs_baseline', 'f1'])
v2_f1 = load_summary_value('results/human_validation/exp11_baseline_v2_vs_human_metrics.json', ['average_vs_baseline', 'f1'])

comparison_rows = [
    {'Prompt': 'V1', 'Context': 'Baseline', 'Avg Kappa': v1_kappa, 'Avg F1': v1_f1},
    {'Prompt': 'V2', 'Context': 'Baseline', 'Avg Kappa': v2_kappa, 'Avg F1': v2_f1},
    {'Prompt': 'V3', 'Context': 'Baseline', 'Avg Kappa': avg_v3_kappa, 'Avg F1': avg_v3_f1},
]
comparison_df = pd.DataFrame(comparison_rows)
print('\nOverall comparison against human annotations:')
display(comparison_df)

per_kc_rows = []
for kc in KC_COLUMNS:
    y_h_a = []
    y_h_b = []
    y_v3 = []
    for pid in common_pids:
        gaps_a = anns_ha.get(pid, set())
        gaps_b = anns_hb.get(pid, set())
        gaps_v3 = anns_v3.get(pid, set())
        y_h_a.append(1 if kc in gaps_a else 0)
        y_h_b.append(1 if kc in gaps_b else 0)
        y_v3.append(1 if kc in gaps_v3 else 0)

    metrics_a = compute_binary_metrics(y_h_a, y_v3)
    metrics_b = compute_binary_metrics(y_h_b, y_v3)
    per_kc_rows.append({
        'KC': kc,
        'Human A Kappa': metrics_a['kappa'],
        'Human B Kappa': metrics_b['kappa'],
        'Avg Kappa': mean([metrics_a['kappa'], metrics_b['kappa']]) if common_pids else 0.0,
        'Human A F1': metrics_a['f1'],
        'Human B F1': metrics_b['f1'],
        'Avg F1': mean([metrics_a['f1'], metrics_b['f1']]) if common_pids else 0.0,
    })

per_kc_df = pd.DataFrame(per_kc_rows).sort_values('Avg Kappa', ascending=True)
per_kc_csv = OUTPUT_DIR / 'v3_per_kc_metrics.csv'
per_kc_df.to_csv(per_kc_csv, index=False)
print(f'Per-KC metrics saved to {per_kc_csv}')
display(per_kc_df)

summary = {
    'num_common_problems': len(common_pids),
    'human_vs_human': res_h_h,
    'v3_vs_human_a': res_a_v3,
    'v3_vs_human_b': res_b_v3,
    'average_vs_v3': {'kappa': avg_v3_kappa, 'f1': avg_v3_f1},
    'comparison_table': comparison_rows,
    'per_kc_metrics': per_kc_rows,
}

summary_path = OUTPUT_DIR / 'exp12_v3_vs_human_metrics.json'
with summary_path.open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(f'Saved summary to {summary_path}')

Common non-empty problems: 44

Overall comparison against human annotations:


,Prompt,Context,Avg Kappa,Avg F1
0,V1,Baseline,0.300067,0.352344
1,V2,Baseline,0.371420,0.412006
2,V3,Baseline,0.423684,0.507523


Per-KC metrics saved to results/human_validation/v3_per_kc_metrics.csv


/mnt/d/Projects/kintsugi/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/mnt/d/Projects/kintsugi/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:991: RuntimeWarning: invalid value encountered in scalar divide
  k = xp.sum(w_mat * confusion) / xp.sum(w_mat * expected)
/mnt/d/Projects/kintsugi/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/mnt/d/Projects/kintsugi/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:991: RuntimeWarning: invalid value encountered in scalar divide
  k = xp.sum(w_mat * confusion) / xp.sum(w_mat * ex

,KC,Human A Kappa,Human B Kappa,Avg Kappa,Human A F1,Human B F1,Avg F1
0,If/Else,0.012821,-0.047619,-0.017399,0.222222,0.166667,0.194444
1,NestedIf,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,NestedFor,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
13,StringLen,0.115578,-0.084507,0.015535,0.200000,0.000000,0.100000
8,LogicCompareNum,0.229759,0.167331,0.198545,0.500000,0.486486,0.493243
17,DefFunction,-0.031250,0.476190,0.222470,0.000000,0.500000,0.250000
7,LogicAndNotOr,0.409091,0.036866,0.222979,0.666667,0.344828,0.505747
3,For,0.530488,0.198675,0.364582,0.631579,0.352941,0.492260
10,StringFormat,0.385093,0.431034,0.408064,0.470588,0.500000,0.485294
5,Math+-*/,0.257384,0.642276,0.449830,0.333333,0.666667,0.500000


Saved summary to results/human_validation/exp14_v3_vs_human_metrics.json
